In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
zip_path = '/content/drive/MyDrive/Datathon/datathon-2026-round-1.zip'
import zipfile
import os

extract_path = '/content/drive/MyDrive/Datathon/data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
!ls /content/drive/MyDrive/Datathon/data

baseline.ipynb	order_items.csv  promotions.csv  sample_submission.csv
customers.csv	orders.csv	 returns.csv	 shipments.csv
geography.csv	payments.csv	 reviews.csv	 web_traffic.csv
inventory.csv	products.csv	 sales.csv


In [ ]:
import pandas as pd

base_path = '/content/drive/MyDrive/Datathon/data/'

orders = pd.read_csv(base_path + 'orders.csv')
order_items = pd.read_csv(base_path + 'order_items.csv')
products = pd.read_csv(base_path + 'products.csv')
customers = pd.read_csv(base_path + 'customers.csv')
returns = pd.read_csv(base_path + 'returns.csv')
web = pd.read_csv(base_path + 'web_traffic.csv')
payments = pd.read_csv(base_path + 'payments.csv')
geography = pd.read_csv(base_path + 'geography.csv')
sales = pd.read_csv(base_path + 'sales.csv')

/tmp/ipykernel_14710/1470763688.py:6: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(base_path + 'order_items.csv')


In [ ]:
orders = pd.read_csv(base_path + 'orders.csv', low_memory=False)
order_items = pd.read_csv(base_path + 'order_items.csv', low_memory=False)
products = pd.read_csv(base_path + 'products.csv', low_memory=False)
customers = pd.read_csv(base_path + 'customers.csv', low_memory=False)
returns = pd.read_csv(base_path + 'returns.csv', low_memory=False)
web = pd.read_csv(base_path + 'web_traffic.csv', low_memory=False)
payments = pd.read_csv(base_path + 'payments.csv', low_memory=False)
geography = pd.read_csv(base_path + 'geography.csv', low_memory=False)
sales = pd.read_csv(base_path + 'sales.csv', low_memory=False)

# Question 1

In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'])

orders = orders.sort_values(['customer_id', 'order_date'])

orders['prev_date'] = orders.groupby('customer_id')['order_date'].shift(1)
orders['gap'] = (orders['order_date'] - orders['prev_date']).dt.days

gaps = orders['gap'].dropna()

gaps.median()

144.0

# Question 2

In [ ]:
products['margin'] = (products['price'] - products['cogs']) / products['price']

products.groupby('segment')['margin'].mean().sort_values(ascending=False)

,margin
segment,
Standard,0.313442
Premium,0.285377
All-weather,0.284176
Activewear,0.265600
Performance,0.263650
Balanced,0.258038
Trendy,0.240758
Everyday,0.236343


#Question 3

In [ ]:
df = returns.merge(products, on='product_id')

df[df['category'] == 'Streetwear']['return_reason'].value_counts()

,count
return_reason,
wrong_size,7626
defective,4330
not_as_described,3854
changed_mind,3830
late_delivery,2159


# Question 4

In [ ]:
web.groupby('traffic_source')['bounce_rate'].mean().sort_values()

,bounce_rate
traffic_source,
email_campaign,0.004458
social_media,0.004476
paid_search,0.004478
referral,0.004499
organic_search,0.004504
direct,0.004511


# Question 5

In [ ]:
pct = order_items['promo_id'].notna().mean()
pct

np.float64(0.3866349316956521)

# Question 6

In [ ]:
df = orders.merge(customers, on='customer_id')
df = df[df['age_group'].notna()]

orders_per_group = df.groupby('age_group')['order_id'].count()
customers_per_group = df.groupby('age_group')['customer_id'].nunique()

ratio = (orders_per_group / customers_per_group).sort_values(ascending=False)
ratio

,0
age_group,
55+,7.268731
45-54,7.220264
35-44,7.206159
25-34,7.112230
18-24,7.068577


# Question 7

In [ ]:
valid_orders = orders[orders['order_status'] == 'delivered']
df = order_items.merge(valid_orders[['order_id', 'zip']], on='order_id')
df = df.merge(geography[['zip', 'region']], on='zip')
df['revenue'] = df['quantity'] * df['unit_price'] - df['discount_amount']

result = df.groupby('region')['revenue'].sum().sort_values(ascending=False)
result

,revenue
region,
East,5.826256e+09
Central,3.762670e+09
West,2.929250e+09


# Question 8

In [ ]:
orders[orders['order_status'] == 'cancelled']['payment_method'].value_counts()

,count
payment_method,
credit_card,28452
cod,15468
paypal,7817
apple_pay,5190
bank_transfer,2535


# Question 9

In [ ]:
df = order_items.merge(products, on='product_id')
total = df.groupby('size').size()

ret = returns.merge(products, on='product_id')
ret_count = ret.groupby('size').size()

rate = (ret_count / total).sort_values(ascending=False)
rate

,0
size,
S,0.056515
L,0.056250
M,0.055660
XL,0.055200


# Question 10

In [ ]:
payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)

,payment_value
installments,
6,24446.654403
3,24399.635486
12,24245.772694
1,24113.274166
2,708.473729
